# Reproducing results on the Sage paper

## Packages

In [ ]:
! pip uninstall sage -y
! pip install -e ../

In [ ]:
import h5py
import numpy as np

import matplotlib.pyplot as plt

## Get Segments for Downloading Real Noise 

In [ ]:
from sage.data.primer import TimelineQuery, get_all_detnames, get_all_runnames

In [ ]:
print(get_all_detnames())
print(get_all_runnames())

In [ ]:
tq = TimelineQuery(detector=["H1", "L1", "V1"], 
                   observing_run=["O3a",],
                   start = 1238166018,
                   end = 1238176018,
                   auto_clean_empty_timelines=True)
                                
tq.download_segments()

In [ ]:
tq.prune_segments(
    rm_short_segments = True,
    rm_min_duration = 22.0,
    rm_allevents = True,
    rm_window_length = 30,
)

In [ ]:
tq.timeline

## Download Segments from GWOSC

In [ ]:
from sage.data.primer import DataReleaseDownloader

In [ ]:
drd = DataReleaseDownloader(
    segments_metadata=tq.timeline,
    save_parent_dir="./",
    noise_low_freq_cutoff = 15.0,
    minimum_segment_duration = 22.0,
    corrupt_trim_length = 0.2,
    max_download_retries = 15,
    retry_delay = 0.5,
    num_workers = 4,
    make_monolithic_file = True,
    sample_rate = 2048.0,
)

drd.download()

In [ ]:
foo = h5py.File('./data_release/data_H1_O3a.h5', 'r')
print(list(foo.keys()))

In [ ]:
foo['segments'].keys()

In [ ]:
# TODO: Find out why this is slightly smaller than expected
num = np.array(foo['segments/00000']).shape[0] + np.array(foo['segments/00001']).shape[0] + np.array(foo['segments/00002']).shape[0]
print((num+0)/2048.)

In [ ]:
foo['segments/00000'].attrs.keys()

## Get Noise PSD Estimates from Real Noise

In [ ]:
from sage.data.noise import GenerateRealNoise
from sage.data.primer import EstimatePSD
from sage.dsp.welch import WelchPSD
from sage.data.primer import NoBlackout, HardRatioBlackout

In [ ]:
noise_gen = GenerateRealNoise("./data_release/data_H1_O3a.h5")
noise_gen(2048)

In [ ]:
class cfg:
    export_dir = "./export_dir"

class data_cfg:
    data_dir = "./data_dir"
    sample_rate = 2048.
    sample_length_in_seconds = 15

epsd = EstimatePSD(
    detector = 'H1',
    num_samples = 10,
    psd_method=WelchPSD(sample_rate=2048., nperseg_in_seconds=4),
    blackout_policy=NoBlackout(),
    store_raw_psds = True,
)

fiducial_psd, freqs = epsd(
    noise_source=noise_gen, 
    cfg=cfg(), 
    data_cfg=data_cfg()
)

plt.plot(freqs, fiducial_psd)
plt.xscale('log')
plt.yscale('log')
plt.xlim(15, 1024)
plt.show()

In [ ]:
class cfg:
    export_dir = "./export_dir"

class data_cfg:
    data_dir = "./data_dir"
    sample_rate = 2048.
    sample_length_in_seconds = 15

epsd = EstimatePSD(
    detector = 'H1',
    num_samples = 10,
    psd_method=WelchPSD(sample_rate=2048., nperseg_in_seconds=4),
    blackout_policy=HardRatioBlackout(4.0),
    store_raw_psds = True,
)

fiducial_psd, freqs = epsd(
    noise_source=noise_gen, 
    cfg=cfg(), 
    data_cfg=data_cfg()
)

plt.plot(freqs, fiducial_psd)
plt.xscale('log')
plt.yscale('log')
plt.xlim(15, 1024)
plt.ylim(1e-49, 1e-39)
plt.show()